# Episode 2B — Multimodal Ingestion: Tables, Charts & Figures

**Course:** Production RAG — YouTube Series  
**GitHub branch:** `episode/02b`

> *"Your DHS reports contain 200+ tables and 100+ charts. The basic pipeline sees none of them correctly. This episode fixes that."*

## The problem we're solving

Our Episode 1–2 pipeline uses PyMuPDF `page.get_text()`. Here's what it produces for Nigeria PR157 Table 3:

```
Table 3 Current fertility
Age group Urban Rural Total 10–14 [0] [2] [1] 15–19 36 114 77 ...
TFR (15–49) 3.9 5.6 4.8 GFR 129 190 160 CBR 28 38 33
```

No column headers attached to data. No structure. A query for **'TFR rural Nigeria'** returns ZERO relevant chunks.

And Figure 2 (the stacked bar chart showing contraceptive trends) is completely **invisible** — not a single character extracted.

## The three-layer solution

| Layer | Tool | Handles | Cost |
|-------|------|---------|------|
| 1 | **Docling** (IBM) | Text + structured tables as Markdown | Free, local |
| 2 | **GPT-4o Vision** | Charts, maps, infographics, figures | ~$0.60 total |
| 3 | **Table-to-Prose** | Makes tables semantically searchable | ~$0.10 total |

## RAGAS impact preview

| Query | Basic pipeline | Multimodal pipeline |
|-------|---------------|--------------------|
| 'TFR in rural Nigeria' | ❌ MISS | ✅ HIT (Table 3, p.42) |
| 'contraceptive trends Nigeria 1990-2024' | ❌ MISS (Figure 2) | ✅ HIT (GPT-4o description) |
| 'family planning demand met vs unmet' | ❌ MISS (bar chart) | ✅ HIT (vision description) |

## 0. Setup & install Docling

In [ ]:
import subprocess, sys
from pathlib import Path

# Install Docling — the key new dependency
print('Installing Docling (first time may take 2-3 minutes to download models)...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', 'docling'],
    capture_output=True, text=True
)
print('✅ Docling installed' if result.returncode == 0 else f'❌ {result.stderr[-300:]}')

cwd = Path().resolve()
repo_root = cwd.parent if cwd.name == 'notebooks' else cwd
import sys; sys.path.insert(0, str(repo_root / 'src'))
from dotenv import load_dotenv; load_dotenv(repo_root / '.env', override=True)
print(f'Repo root: {repo_root}')

## 1. The problem — what basic extraction misses

Run this cell while showing the actual PDF page on screen.
Point at each artefact as you describe it.

In [ ]:
# Show raw PyMuPDF output for the page containing Table 3 and Figure 2
import pymupdf

nigeria_pdf = repo_root / 'data' / 'raw' / 'PR157.pdf'
doc = pymupdf.open(str(nigeria_pdf))

# Find a page with a table (search for 'TFR' in pages)
table_page_num = None
for i, page in enumerate(doc):
    text = page.get_text()
    if 'TFR' in text and 'Urban' in text and 'Rural' in text:
        table_page_num = i
        break

if table_page_num is not None:
    page = doc[table_page_num]
    raw_text = page.get_text('text')
    print(f'=== PyMuPDF raw extraction — Page {table_page_num + 1} ===')
    print(raw_text[:2000])
    print()
    print('❌ PROBLEMS:')
    print('  - Table columns collapsed into single line')
    print('  - Column headers (Urban/Rural/Total) detached from data rows')
    print('  - Row labels (Age group, TFR, GFR, CBR) mixed into the data')
    print('  - Query "TFR rural Nigeria" will NOT match this text')
else:
    print('Table page not found — check PDF')
doc.close()

In [ ]:
# Count how many figures are in the document that PyMuPDF completely misses
doc = pymupdf.open(str(nigeria_pdf))
figure_count = 0
chart_mentions = 0

for page in doc:
    blocks = page.get_text('blocks')
    for block in blocks:
        if block[6] == 1:  # block type 1 = image
            figure_count += 1
    if 'Figure' in page.get_text():
        chart_mentions += 1

doc.close()
print(f'Images/figures in PR157.pdf: {figure_count}')
print(f'Pages mentioning "Figure":   {chart_mentions}')
print()
print('PyMuPDF get_text() recovers: 0 of these figures')
print('Every chart, map, and infographic is INVISIBLE to the basic pipeline.')
print('That includes Figure 1 (TFR trends), Figure 2 (family planning), etc.')

## 2. Layer 1 — Docling: structured table extraction

Docling uses **DocLayNet** (layout analysis) and **TableFormer** (table structure).
On camera: open the Docling GitHub (github.com/docling-project/docling), show the star count,
explain it's IBM Research — production-grade, not a weekend project.

In [ ]:
# Run Docling on just the first 20 pages for speed during demo
# Full corpus run takes ~5-10 min
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption
import time

# Configure — enable picture extraction for Layer 2
pipeline_opts = PdfPipelineOptions()
pipeline_opts.generate_picture_images = True
pipeline_opts.images_scale = 2.0

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_opts)}
)

print('Running Docling on Nigeria DHS (first 30 pages)...')
t0 = time.perf_counter()

# Note: for demo we use the full PDF but Docling is fast
result = converter.convert(str(nigeria_pdf))
doc    = result.document
elapsed = time.perf_counter() - t0

print(f'✅ Docling complete in {elapsed:.1f}s')
print(f'Pages processed: {len(doc.pages)}')

In [ ]:
# Count element types
texts, tables, figures, headings = [], [], [], []

for item, level in doc.iterate_items():
    item_type = type(item).__name__
    if item_type in ('TextItem', 'ParagraphItem'):  texts.append(item)
    elif item_type == 'TableItem':                   tables.append(item)
    elif item_type in ('PictureItem', 'FigureItem'): figures.append(item)
    elif item_type == 'SectionHeaderItem':           headings.append(item)

print('Docling element breakdown:')
print(f'  Text paragraphs: {len(texts)}')
print(f'  Tables:          {len(tables)}')
print(f'  Figures:         {len(figures)}')
print(f'  Headings:        {len(headings)}')
print(f'  Total:           {len(texts)+len(tables)+len(figures)+len(headings)}')

In [ ]:
# THE KEY DEMO: Show Docling's table output vs PyMuPDF
print('=== DOCLING — Table extraction (first 3 tables) ===')
print()
for i, table in enumerate(tables[:3], 1):
    md = table.export_to_markdown()
    print(f'--- Table {i} (Markdown) ---')
    print(md[:800])
    print()

print('✅ DIFFERENCE vs PyMuPDF:')
print('  - Column headers ATTACHED to their data columns')
print('  - Row labels on left, values aligned to correct columns')
print('  - Markdown format: structured, parseable, embeddable')
print('  - "TFR rural Nigeria" query will NOW match this text')

## 3. Layer 3 — Table-to-Prose

Even perfect Markdown tables have poor semantic embeddings.
`| 3.9 | 5.6 | 4.8 |` is not close to `total fertility rate urban rural national`.
We generate a prose version for semantic retrieval.

In [ ]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def table_to_prose(markdown: str, context: str = '') -> str:
    prompt = f"""Convert this Markdown table from a DHS health report into clear, descriptive prose.
Report context: {context}

Table:
{markdown}

Write 2-5 sentences that state what the table shows and include ALL specific numerical values 
with their row/column context. Use language a health researcher would use when asking about this data.
Output ONLY the prose, nothing else."""
    
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=300, temperature=0,
    )
    return resp.choices[0].message.content.strip()

# Demo on the first table
if tables:
    md   = tables[0].export_to_markdown()
    prose = table_to_prose(md, 'Nigeria DHS 2021')
    print('=== MARKDOWN (Layer 1 output) ===')
    print(md[:600])
    print()
    print('=== PROSE (Layer 3 output — semantically searchable) ===')
    print(prose)
    print()
    print('Cost: ~$0.001 per table × ~50 tables = ~$0.05 for entire corpus')

## 4. Layer 2 — GPT-4o Vision: figures and charts

This is the most impressive part. Figure 2 from Nigeria PR157 shows
a stacked bar chart of family planning demand 1990–2024.
The basic pipeline sees nothing. GPT-4o reads every number.

In [ ]:
import base64, io

def describe_figure(image_b64: str, caption: str = '', context: str = '') -> str:
    prompt = f"""You are analysing a figure from a Demographic and Health Survey (DHS) report.
Report context: {context}
Figure caption: {caption}

Describe this figure comprehensively for a health data retrieval system:
1. CHART TYPE: What kind of visualisation?
2. TITLE/SUBJECT: What does it show? Include all axis labels and legend items.
3. DATA VALUES: List ALL specific numbers and percentages visible.
4. TIME PERIOD: What years or periods are shown?
5. KEY TRENDS: Main patterns or findings?

Be specific — include ALL numerical values."""
    
    resp = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': [
            {'type': 'text', 'text': prompt},
            {'type': 'image_url', 'image_url': {
                'url': f'data:image/png;base64,{image_b64}',
                'detail': 'high'
            }}
        ]}],
        max_tokens=800,
    )
    return resp.choices[0].message.content.strip()

# Process figures Docling found
vision_results = []
for i, fig in enumerate(figures[:3]):
    try:
        if hasattr(fig, 'image') and fig.image:
            img = fig.image.pil_image
            buf = io.BytesIO()
            img.save(buf, format='PNG')
            b64 = base64.b64encode(buf.getvalue()).decode()
            
            caption = str(fig.caption) if hasattr(fig, 'caption') and fig.caption else ''
            print(f'--- Figure {i+1} ---')
            print(f'Caption: {caption[:100] if caption else "(no caption)"}')
            description = describe_figure(b64, caption, 'Nigeria DHS 2021')
            print(f'GPT-4o description:')
            print(description)
            vision_results.append({'fig': i+1, 'description': description})
            print()
        else:
            print(f'Figure {i+1}: No image data extracted by Docling')
    except Exception as e:
        print(f'Figure {i+1} error: {e}')

In [ ]:
# Fallback: render a page with PyMuPDF and send the whole page to GPT-4o
# Use this when Docling doesn't extract figure images

def render_page_and_describe(pdf_path, page_num: int, context: str = '') -> str:
    """Render page as image, send to GPT-4o, get description."""
    import pymupdf
    doc  = pymupdf.open(str(pdf_path))
    page = doc[page_num - 1]
    pix  = page.get_pixmap(dpi=150)      # 150 DPI good balance quality/cost
    img_bytes = pix.tobytes('png')
    doc.close()
    b64 = base64.b64encode(img_bytes).decode()
    return describe_figure(b64, context=context)

# Find a page with a figure reference
doc2 = pymupdf.open(str(nigeria_pdf))
fig_page = None
for i, page in enumerate(doc2):
    if 'Figure 2' in page.get_text() and 'family planning' in page.get_text().lower():
        fig_page = i + 1
        break
doc2.close()

if fig_page:
    print(f'Found Figure 2 on page {fig_page}')
    print('Sending page to GPT-4o vision...')
    description = render_page_and_describe(nigeria_pdf, fig_page, 'Nigeria DHS 2021')
    print()
    print('=== GPT-4o Vision Description of Figure 2 ===')
    print(description)
else:
    print('Figure 2 page not found — try adjusting the search text')

## 5. Full multimodal pipeline — using MultimodalLoader

In [ ]:
from rag.ingestion.multimodal_loader import MultimodalLoader, element_stats
import json

meta_map = json.loads((repo_root / 'data' / 'metadata.json').read_text())

# Run multimodal loader on Nigeria only (fastest for demo)
loader = MultimodalLoader(
    vision_enabled=True,    # set False to skip GPT-4o (cheaper, for testing)
    table_prose=True,       # generate prose versions of tables
)

meta = meta_map.get('PR157', {})
print('Running full multimodal pipeline on Nigeria DHS...')
elements = loader.load_pdf(
    repo_root / 'data' / 'raw' / 'PR157.pdf',
    country=meta.get('country',''),
    year=meta.get('year',''),
    report_type=meta.get('report_type','dhs'),
    report_title=meta.get('report_title',''),
)

stats = element_stats(elements)
print()
print('=== Multimodal Load Statistics ===')
for k, v in stats.items():
    print(f'  {k:<20} {v}')

In [ ]:
# Convert to Documents and show the breakdown
docs = loader.to_documents(elements)
print(f'Total LangChain Documents: {len(docs)}')
print()

tables_docs  = [d for d in docs if d.metadata['element_type'] == 'table']
prose_docs   = [d for d in docs if d.metadata.get('source_element') == 'table']
figure_docs  = [d for d in docs if d.metadata['has_figure']]
text_docs    = [d for d in docs if d.metadata['element_type'] == 'text']

print(f'Text documents:       {len(text_docs)}')
print(f'Table (Markdown):     {len(tables_docs)}')
print(f'Table (Prose):        {len(prose_docs)}')
print(f'Figure (described):   {len(figure_docs)}')
print()
print('Note: tables produce TWO documents — Markdown + Prose')
print('This gives both exact structural retrieval AND semantic retrieval')

In [ ]:
# Show a table document (Markdown) and its prose twin
if tables_docs:
    print('=== TABLE DOCUMENT (Markdown — exact retrieval) ===')
    print(tables_docs[0].page_content[:400])
    print()

if prose_docs:
    print('=== TABLE PROSE DOCUMENT (semantic retrieval) ===')
    print(prose_docs[0].page_content)
    print()

if figure_docs:
    print('=== FIGURE DOCUMENT (GPT-4o description) ===')
    print(figure_docs[0].page_content[:500])
    print(f'  confidence: {figure_docs[0].metadata["confidence"]}')

## 6. RAGAS comparison — does it actually improve retrieval?

In [ ]:
# Semantic similarity test: can we find the table with a natural language query?
from rag.ingestion.embedder import Embedder, top_similar

emb = Embedder()
query = 'total fertility rate in rural Nigeria'

# Old pipeline: flat text
old_texts = [
    'Table 3 Current fertility Age group Urban Rural Total 10-14 0 2 1 15-19 36 114 77 TFR 3.9 5.6 4.8',
]

# New pipeline: prose
new_texts = [
    prose_docs[0].page_content if prose_docs else '',
    tables_docs[0].page_content if tables_docs else '',
]

from rag.ingestion.embedder import cosine_similarity

q_vec = emb.embed_query(query)

print(f'Query: "{query}"')
print()
print('Old pipeline (raw PyMuPDF text):')
for t in old_texts:
    if t:
        sim = cosine_similarity(q_vec, emb.embed_query(t))
        print(f'  similarity: {sim:.4f}  →  {t[:80]}...')

print()
print('New pipeline (Docling prose + Markdown):')
for t in new_texts:
    if t:
        sim = cosine_similarity(q_vec, emb.embed_query(t[:200]))
        print(f'  similarity: {sim:.4f}  →  {t[:80]}...')

print()
print('Higher similarity = more likely to be retrieved for this query.')
print('This is context precision improving before we even run RAGAS.')

## 7. Update the ingestion pipeline — run full multimodal ingest

In [ ]:
# The updated ingest command with multimodal flag
print('To re-ingest with multimodal support:')
print()
print('python scripts/ingest.py --metadata data/metadata.json --multimodal')
print()
print('Or make ingest-multimodal')
print()
print('Time estimate for full corpus:')
print('  Docling extraction: ~5-10 min (CPU-only)')
print('  GPT-4o vision:      ~2-3 min  (~200 figures × 1 sec each)')
print('  Table prose:        ~1-2 min  (~100 tables × 0.5 sec each)')
print('  Embedding + upsert: ~2-3 min  (same as before)')
print('  Total: ~15-20 min vs ~2 min for basic pipeline')
print()
print('Cost: ~$0.70 total vs ~$0.60 for basic (table prose + figure vision)')
print('This runs ONCE — re-ingestion is idempotent (content_hash dedup)')

## ✅ Episode 2B complete

| What we added | How it works | Impact |
|--------------|-------------|--------|
| Docling extraction | DocLayNet + TableFormer → structured Markdown | Tables now correctly structured |
| GPT-4o vision | Page rendered → GPT-4o describes → stored as text | Charts and maps now searchable |
| Table-to-prose | GPT-4o-mini converts Markdown → natural language | Semantic queries now hit tables |
| Dual indexing | Tables stored as BOTH Markdown AND prose | Maximum retrieval coverage |

**Expected RAGAS improvement (Episode 9 will measure this):**
- Context Precision: +15-25% (correct chunks retrieved for table queries)
- Context Recall: +10-20% (chart data now in corpus)
- Faithfulness: +5-10% (answers grounded in actual table numbers, not hallucinated)

## What's next — Episode 3

**Embeddings** — now that we have richer content (tables as prose, figures as text),
we'll see how the embedding model handles domain-specific health vocabulary.
Does `text-embedding-3-small` know that 'MMR' = 'maternal mortality ratio'?

**GitHub branch:** `episode/02b`